In [ ]:
!pip install mlflow --quiet

import os
MLFLOW_URI = "file:///tmp/mlruns" if "COLAB_RELEASE_TAG" in os.environ else "http://mlflow:5000"


# ML Basics: Training Your First Model

This notebook mirrors the **ML Basics** guide at [learnmlops.ops4life.com/guides/ml-basics/](https://learnmlops.ops4life.com/guides/ml-basics/).

Train a Random Forest classifier on the Iris dataset, log the experiment with MLflow, and inspect the results in the tracking UI. This is the starting point of the MLOps journey — before pipelines, before production.

**What we'll cover:**
1. Load the Iris dataset
2. Split into train and test sets
3. Train with MLflow experiment tracking
4. Inspect results

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment('iris-classification')

## Step 1: Load the Iris dataset

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names
class_names = iris.target_names

print(f'Dataset: {X.shape[0]} samples, {X.shape[1]} features')
print(f'Classes: {list(class_names)}')
print(f'Features: {list(feature_names)}')

## Step 2: Split into train and test sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]} samples')
print(f'Test:  {X_test.shape[0]} samples')

## Step 3: Train with MLflow tracking

In [ ]:
with mlflow.start_run(run_name='rf-baseline'):
    n_estimators = 100
    max_depth = 5

    mlflow.log_param('n_estimators', n_estimators)
    mlflow.log_param('max_depth', max_depth)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    mlflow.log_metric('accuracy', acc)

    mlflow.sklearn.log_model(model, 'model')

    run_id = mlflow.active_run().info.run_id
    print(f'Run ID: {run_id}')
    print(f'Accuracy: {acc:.4f}')

MLflow recorded the run parameters (`n_estimators`, `max_depth`), the accuracy metric, and the serialised model artifact. Open the MLflow UI at [mlflow.learnmlops.ops4life.com](https://mlflow.learnmlops.ops4life.com) and navigate to the **iris-classification** experiment to see the run.

## Step 4: Inspect results

In [ ]:
print(classification_report(y_test, y_pred, target_names=class_names))

## Next Steps

- Explore the MLflow UI: compare runs, view logged artifacts, and register a model
- Continue to the pipeline: `02-pipeline/dataset-pipeline/ingest.ipynb`
- Full guide: [learnmlops.ops4life.com/guides/ml-basics/](https://learnmlops.ops4life.com/guides/ml-basics/)